# 12｜深層神經網路訓練_梯度與遷移

用梯度範數與深度尺度理解消失、爆炸與裁剪。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")


## 課堂小實驗

In [ ]:
depth = np.array([1, 2, 5, 10, 20])
sigmoid_upper = 0.25 ** depth
scale_table = pd.DataFrame({'depth': depth, 'sigmoid_derivative_upper_bound': sigmoid_upper})

gradient = np.array([3.0, 4.0, 12.0])
before = np.linalg.norm(gradient)
clip_norm = 5.0
clipped = gradient * min(1.0, clip_norm / before)
after = np.linalg.norm(clipped)
print(f'gradient norm: {before:.1f} -> {after:.1f}')
scale_table

## 執行結果圖

In [ ]:
plt.semilogy(depth, sigmoid_upper, 'o-', label=r'$(1/4)^L$')
plt.axhline(1e-6, color='tab:red', ls='--', label='1e-6')
plt.xlabel('number of saturated sigmoid layers')
plt.ylabel('upper bound of derivative product')
plt.title('Repeated small derivatives shrink the backward signal')
plt.legend()
savefig('12_gradient_transfer_demo')
report('deep_gradient_transfer', {'gradient_norm_before': before, 'gradient_norm_after': after,
                                  'depths': depth.tolist(), 'sigmoid_upper': sigmoid_upper.tolist()})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。